# RNNs for Language Modeling

### Learning Objectives:

- Understand the motivation behind using Recurrent Neural Networks in NLP.
- Explore key concepts: sequence modeling, recurrence, hidden states, backpropagation through time (briefly).
- Implement RNN architectures (Simple RNN, LSTM) using PyTorch.
- Train an RNN-based language model on real-world data.
- Evaluate and visualize the results.

Recurrent Neural Networks (RNNs) are specifically designed to handle sequential data—data where the order matters, such as text, audio, or time-series data. Unlike traditional neural networks that process input independently, RNNs maintain a hidden state that captures information about previous inputs, allowing them to model sequences effectively.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from transformers import default_data_collator
import numpy as np

from datasets import load_dataset

: 

## Dataset: Loading & Preprocessing

We use the WikiText-2 dataset, a practical benchmark for language modeling tasks.

In [ ]:
datasets = load_dataset("wikitext", "wikitext-2-raw-v1")
datasets

We then need to tokenize our dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokenized_datasets = datasets.map(
    lambda x: tokenizer(x["text"], return_attention_mask=False),
    batched=True
)

print(tokenized_datasets)

In [ ]:
tokenized_datasets["train"].to_pandas().head()


## Preparing Data for Language Modeling

Preparing Data for Language Modeling. By creating sequences of fixed lenght.

In [ ]:
block_size = 64

def group_texts(examples):
    concatenated = sum(examples["input_ids"], [])
    total_length = (len(concatenated) // block_size) * block_size
    input_ids = [concatenated[i:i + block_size] for i in range(0, total_length, block_size)]
    labels = [ids[1:] + [tokenizer.eos_token_id] for ids in input_ids]
    return {"input_ids": input_ids, "labels": labels}

lm_dataset = tokenized_datasets.map(group_texts, batched=True, remove_columns=["input_ids", "text"])

train_loader = DataLoader(lm_dataset["train"], batch_size=32, shuffle=True, collate_fn=default_data_collator)
valid_loader = DataLoader(lm_dataset["validation"], batch_size=32, collate_fn=default_data_collator)


The labels represent the next tokens that the model is trained to predict, which are essentially the input sequence shifted by one position.

In [ ]:
lm_dataset["train"].to_pandas().head()

### Embedding Model

We can either train our own embeddings or load the weights of a pre-trained model. Here, we load the weights of the GPT-2 model. Note that the embedding model must match the tokenizer model to ensure compatibility.

In [ ]:
# Load GPT-2 pretrained embeddings
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
pretrained_model = AutoModel.from_pretrained(model_name)
pretrained_embeddings = pretrained_model.get_input_embeddings().weight

A simple language model takes tokenized inputs, extracts their embeddings, processes them through a recurrent layer, and finally passes them to a feed-forward layer to compute the probabilities of the next token across the vocabulary.

In [ ]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        vocab_size, embedding_dim = pretrained_embeddings.shape
        self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=True)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        lstm_out, _ = self.lstm(embeddings)
        logits = self.fc(lstm_out)
        return logits


### Training loop

In [ ]:
vocab_size = tokenizer.vocab_size
embedding_dim = 32
hidden_dim = 64

model = LSTMLanguageModel(vocab_size, embedding_dim, hidden_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids).permute(0, 2, 1)  # (B, vocab, seq_len)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids).permute(0, 2, 1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    return total_loss / len(loader)


In [ ]:
epochs = 3

for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader)
    val_loss = evaluate(model, valid_loader)

    print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Perplexity: {np.exp(val_loss):.2f}")


### Generating Text

In [ ]:
def generate(model, tokenizer, prompt, length=50):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    generated = input_ids
    with torch.no_grad():
        for _ in range(length):
            logits = model(generated)
            next_token_logits = logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            generated = torch.cat((generated, next_token), dim=1)

    return tokenizer.decode(generated[0])

print(generate(model, tokenizer, prompt="Generative AI"))
